In [ ]:
# =========================================================
# 0. Library Import & Environment Setup
# =========================================================
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

try:
    import shap
    SHAP_AVAILABLE = True
except:
    SHAP_AVAILABLE = False

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/FinGuard"
DATA_PATH = f"{BASE_PATH}/card_transdata.csv"
IMAGE_PATH = f"{BASE_PATH}/images"
MODEL_PATH = f"{BASE_PATH}/models"

os.makedirs(IMAGE_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

plt.rcParams["figure.dpi"] = 120

print("BASE_PATH:", BASE_PATH)
print("DATA_PATH:", DATA_PATH)

Mounted at /content/drive
BASE_PATH: /content/drive/MyDrive/FinGuard
DATA_PATH: /content/drive/MyDrive/FinGuard/card_transdata.csv


In [ ]:
# =========================================================
# 1. Data Loading
# =========================================================
df = pd.read_csv(DATA_PATH)

print("데이터 크기:", df.shape)
display(df.head())
print(df.columns.tolist())

데이터 크기: (1000000, 8)


,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip', 'used_pin_number', 'online_order', 'fraud']


In [ ]:
# =========================================================
# 2. Column Setting
# =========================================================
target_col = "fraud"

all_feature_cols = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order"
]

print("전체 feature:", all_feature_cols)

전체 feature: ['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip', 'used_pin_number', 'online_order']


In [ ]:
# =========================================================
# 3. Experiment Feature Sets
# =========================================================
feature_sets = {
    "service_aligned": [
        "distance_from_home",
        "distance_from_last_transaction",
        "ratio_to_median_purchase_price",
        "repeat_retailer",
        "used_chip"
    ],

    "drop_ratio_from_service": [
        "distance_from_home",
        "distance_from_last_transaction",
        "repeat_retailer",
        "used_chip"
    ],

    "drop_ratio_distance_from_service": [
        "distance_from_last_transaction",
        "repeat_retailer",
        "used_chip"
    ]
}

for exp_name, cols in feature_sets.items():
    print(f"{exp_name}: {cols}")

service_aligned: ['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip']
drop_ratio_from_service: ['distance_from_home', 'distance_from_last_transaction', 'repeat_retailer', 'used_chip']
drop_ratio_distance_from_service: ['distance_from_last_transaction', 'repeat_retailer', 'used_chip']


In [ ]:
# =========================================================
# 4. Common Train/Test Split Index
# =========================================================
X_all = df[all_feature_cols].copy()
y_all = df[target_col].copy()

X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    X_all.index,
    y_all,
    test_size=0.2,
    random_state=SEED,
    stratify=y_all
)

print("train size:", len(X_train_idx))
print("test size :", len(X_test_idx))
print("\n[Train class ratio]")
print(y_train.value_counts(normalize=True).round(4))
print("\n[Test class ratio]")
print(y_test.value_counts(normalize=True).round(4))

train size: 800000
test size : 200000

[Train class ratio]
fraud
0.0    0.9126
1.0    0.0874
Name: proportion, dtype: float64

[Test class ratio]
fraud
0.0    0.9126
1.0    0.0874
Name: proportion, dtype: float64


In [ ]:
# =========================================================
# 5. Utility Function
# =========================================================
def run_experiment(exp_name, feature_cols, final_threshold=0.8, save_artifacts=False):
    print("=" * 80)
    print(f"[Experiment] {exp_name}")
    print("사용 feature:", feature_cols)

    X = df[feature_cols].copy()
    y = df[target_col].copy()

    X_train = X.loc[X_train_idx]
    X_test = X.loc[X_test_idx]

    # ---------------------------
    # Baseline model
    # ---------------------------
    baseline_model = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=SEED
    )

    baseline_model.fit(X_train, y_train)

    y_proba_base = baseline_model.predict_proba(X_test)[:, 1]
    y_pred_base = (y_proba_base >= 0.5).astype(int)

    baseline_metrics = {
        "Experiment": exp_name,
        "Setting": "Baseline",
        "Threshold": 0.5,
        "Accuracy": accuracy_score(y_test, y_pred_base),
        "Precision": precision_score(y_test, y_pred_base, zero_division=0),
        "Recall": recall_score(y_test, y_pred_base, zero_division=0),
        "F1": f1_score(y_test, y_pred_base, zero_division=0),
    }

    # ---------------------------
    # Resampling
    # ---------------------------
    resampling_pipeline = Pipeline([
        ("smote", SMOTE(random_state=SEED)),
        ("under", RandomUnderSampler(random_state=SEED))
    ])

    X_resampled, y_resampled = resampling_pipeline.fit_resample(X_train, y_train)

    # ---------------------------
    # Resampled model
    # ---------------------------
    resampled_model = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=SEED
    )

    resampled_model.fit(X_resampled, y_resampled)

    y_proba_resampled = resampled_model.predict_proba(X_test)[:, 1]

    # threshold sweep
    thresholds = np.arange(0.1, 1.0, 0.1)
    threshold_results = []

    for t in thresholds:
        y_pred_t = (y_proba_resampled >= t).astype(int)
        threshold_results.append({
            "Experiment": exp_name,
            "Threshold": round(float(t), 2),
            "Accuracy": accuracy_score(y_test, y_pred_t),
            "Precision": precision_score(y_test, y_pred_t, zero_division=0),
            "Recall": recall_score(y_test, y_pred_t, zero_division=0),
            "F1": f1_score(y_test, y_pred_t, zero_division=0),
        })

    threshold_df = pd.DataFrame(threshold_results)

    # final threshold
    y_pred_final = (y_proba_resampled >= final_threshold).astype(int)

    final_metrics = {
        "Experiment": exp_name,
        "Setting": "Final Model",
        "Threshold": final_threshold,
        "Accuracy": accuracy_score(y_test, y_pred_final),
        "Precision": precision_score(y_test, y_pred_final, zero_division=0),
        "Recall": recall_score(y_test, y_pred_final, zero_division=0),
        "F1": f1_score(y_test, y_pred_final, zero_division=0),
    }

    # summary
    summary_df = pd.DataFrame([
        baseline_metrics,
        final_metrics
    ])

    print("\n[Summary]")
    display(summary_df.round(4))

    print("\n[Threshold Results]")
    display(threshold_df.round(4))

    print("\n[Final Classification Report]")
    print(classification_report(y_test, y_pred_final, zero_division=0))

    # feature importance
    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": resampled_model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\n[Feature Importance]")
    display(importance_df.round(6))

    # save plots optionally
    if save_artifacts:
        exp_image_path = f"{IMAGE_PATH}/{exp_name}"
        os.makedirs(exp_image_path, exist_ok=True)

        # threshold plot
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(threshold_df["Threshold"], threshold_df["Precision"], marker="o", label="Precision")
        ax.plot(threshold_df["Threshold"], threshold_df["Recall"], marker="o", label="Recall")
        ax.plot(threshold_df["Threshold"], threshold_df["F1"], marker="o", label="F1")
        ax.set_title(f"{exp_name} - Precision / Recall / F1 by Threshold")
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Score")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.savefig(f"{exp_image_path}/threshold_metrics.png", bbox_inches="tight")
        plt.show()

        # confusion matrix
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred_final, ax=ax, values_format="d"
        )
        ax.set_title(f"{exp_name} - Confusion Matrix (threshold={final_threshold})")
        plt.tight_layout()
        plt.savefig(f"{exp_image_path}/confusion_matrix.png", bbox_inches="tight")
        plt.show()

        # feature importance
        plot_df = importance_df.head(10).sort_values("importance", ascending=True)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(plot_df["feature"], plot_df["importance"])
        ax.set_title(f"{exp_name} - Feature Importance")
        ax.set_xlabel("Importance")
        plt.tight_layout()
        plt.savefig(f"{exp_image_path}/feature_importance.png", bbox_inches="tight")
        plt.show()

        # SHAP
        if SHAP_AVAILABLE:
            sample_X = X_test.sample(min(500, len(X_test)), random_state=SEED)
            explainer = shap.TreeExplainer(resampled_model)
            shap_values = explainer.shap_values(sample_X)

            shap.summary_plot(shap_values, sample_X, show=False)
            plt.tight_layout()
            plt.savefig(f"{exp_image_path}/shap_summary.png", bbox_inches="tight")
            plt.show()

    return summary_df, threshold_df, importance_df

In [ ]:
# =========================================================
# 6. Run Experiments
# =========================================================
results_summary = []
results_threshold = []

for exp_name, cols in feature_sets.items():
    summary_df, threshold_df, importance_df = run_experiment(
        exp_name=exp_name,
        feature_cols=cols,
        final_threshold=0.8,
        save_artifacts=False   # 필요하면 True
    )

    results_summary.append(summary_df)
    results_threshold.append(threshold_df)

all_summary_df = pd.concat(results_summary, ignore_index=True)
all_threshold_df = pd.concat(results_threshold, ignore_index=True)

print("\n[All Summary]")
display(all_summary_df.round(4))

[Experiment] full
사용 feature: ['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip', 'used_pin_number', 'online_order']

[Summary]


,Experiment,Setting,Threshold,Accuracy,Precision,Recall,F1
0,full,Baseline,0.5,0.9984,0.9916,0.9903,0.9910
1,full,Final Model,0.8,0.9985,0.9941,0.9883,0.9912



[Threshold Results]


,Experiment,Threshold,Accuracy,Precision,Recall,F1
0,full,0.1,0.9970,0.9671,0.9999,0.9833
1,full,0.2,0.9971,0.9682,0.9998,0.9838
2,full,0.3,0.9972,0.9689,0.9994,0.9839
3,full,0.4,0.9972,0.9703,0.9989,0.9844
4,full,0.5,0.9973,0.9715,0.9986,0.9849
5,full,0.6,0.9973,0.9727,0.9976,0.9850
6,full,0.7,0.9976,0.9778,0.9955,0.9866
7,full,0.8,0.9985,0.9941,0.9883,0.9912
8,full,0.9,0.9986,1.0000,0.9844,0.9922



[Final Classification Report]
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    182519
         1.0       0.99      0.99      0.99     17481

    accuracy                           1.00    200000
   macro avg       1.00      0.99      1.00    200000
weighted avg       1.00      1.00      1.00    200000


[Feature Importance]


...

,Experiment,Threshold,Accuracy,Precision,Recall,F1
0,drop_ratio_distance,0.1,0.4691,0.1392,0.9792,0.2438
1,drop_ratio_distance,0.2,0.4694,0.1393,0.9789,0.2439
2,drop_ratio_distance,0.3,0.4703,0.1392,0.9763,0.2437
3,drop_ratio_distance,0.4,0.4718,0.1393,0.9740,0.2438
4,drop_ratio_distance,0.5,0.4739,0.1395,0.9712,0.2440
5,drop_ratio_distance,0.6,0.6854,0.1676,0.6552,0.2669
6,drop_ratio_distance,0.7,0.9179,0.9184,0.0670,0.1249
7,drop_ratio_distance,0.8,0.9179,0.9184,0.0670,0.1249
8,drop_ratio_distance,0.9,0.9180,1.0000,0.0618,0.1165



[Final Classification Report]
              precision    recall  f1-score   support

         0.0       0.92      1.00      0.96    182519
         1.0       0.92      0.07      0.12     17481

    accuracy                           0.92    200000
   macro avg       0.92      0.53      0.54    200000
weighted avg       0.92      0.92      0.88    200000


[Feature Importance]


,feature,importance
4,online_order,0.624501
3,used_pin_number,0.229381
1,repeat_retailer,0.060987
2,used_chip,0.055587
0,distance_from_last_transaction,0.029544



[All Summary]


,Experiment,Setting,Threshold,Accuracy,Precision,Recall,F1
0,full,Baseline,0.5,0.9984,0.9916,0.9903,0.9910
1,full,Final Model,0.8,0.9985,0.9941,0.9883,0.9912
2,drop_ratio,Baseline,0.5,0.9367,0.9748,0.2833,0.4391
3,drop_ratio,Final Model,0.8,0.9362,0.9567,0.2831,0.4369
4,drop_ratio_distance,Baseline,0.5,0.9181,0.9477,0.0663,0.1239
5,drop_ratio_distance,Final Model,0.8,0.9179,0.9184,0.0670,0.1249


In [ ]:
# =========================================================
# 8. Save Comparison Results
# =========================================================
all_summary_df.to_csv(f"{BASE_PATH}/feature_set_summary.csv", index=False)
all_threshold_df.to_csv(f"{BASE_PATH}/feature_set_thresholds.csv", index=False)

print("Saved:")
print(f"- {BASE_PATH}/feature_set_summary.csv")
print(f"- {BASE_PATH}/feature_set_thresholds.csv")

Saved:
- /content/drive/MyDrive/FinGuard/feature_set_summary.csv
- /content/drive/MyDrive/FinGuard/feature_set_thresholds.csv
